In [15]:
import pandas as pd
import numpy as np

In [13]:
bonds = pd.read_csv('bonds.csv', skiprows=1)
events = pd.read_csv('events.csv')

In [16]:
bonds.head() 

,BondID,Coupon,Frequency,MonthsSinceCoupon
0,BOND1,0.05,2,3
1,BOND2,0.04,1,6
2,BOND3,0.06,2,4
3,BOND4,0.03,1,6
4,BOND5,0.07,2,1


In [17]:
events.head()

,EventID,Desk,Trader,BondID,BuySell,Quantity,CleanPrice
0,1,NY,T_NY_2,BOND1,SELL,50,99.37
1,2,NY,T_NY_2,BOND5,SELL,60,106.56
2,3,NY,T_NY_1,BOND3,SELL,40,103.46
3,4,LN,T_LN_1,BOND5,SELL,20,105.77
4,5,HK,T_HK_2,BOND1,BUY,20,99.64


Metrics: 
- Position
- Accured Intested 
- Dirty Price
- Present value 
- Change in present value 



show bond
metrics, position, clean price, dirty price, PV
pnl bond



In [20]:
# AI = 100 * Coupon * MonthsSinceCoupon / 12
bonds['AI'] = 100 * (bonds['Coupon'] / bonds['Frequency']) * bonds['MonthsSinceCoupon'] / 12
ai = bonds.set_index('BondID')['AI'].to_dict()

bonds_list = [f'BOND{i}' for i in range(1, 6)]

# Running state
positions    = {b: 0   for b in bonds_list}
dirty_prices = {b: 0.0 for b in bonds_list}

rows = []
for _, evt in events.iterrows():
    bond  = evt['BondID']
    qty   = evt['Quantity'] if evt['BuySell'] == 'BUY' else -evt['Quantity']

    positions[bond]    += qty
    dirty_prices[bond]  = evt['CleanPrice'] + ai[bond]

    row = {'EventID': evt['EventID']}
    for i, b in enumerate(bonds_list, start=1):
        s = f'stock{i}'
        row[f'{s}position']   = positions[b]
        row[f'{s}dirtyprice'] = round(dirty_prices[b], 4)
        row[f'{s}PV']         = round(dirty_prices[b] * positions[b], 4)
    rows.append(row)

df = pd.DataFrame(rows).set_index('EventID')

# P&L = change in PV from previous event (event 0 baseline is all-zero)
for i in range(1, 6):
    pv_col  = f'stock{i}PV'
    pnl_col = f'stock{i}P&L'
    df[pnl_col] = df[pv_col].diff()
    df.loc[df.index[0], pnl_col] = df.loc[df.index[0], pv_col]  # first event: Δ from 0

# Order columns: position → dirtyprice → PV → P&L, repeated per stock
ordered_cols = []
for i in range(1, 6):
    s = f'stock{i}'
    ordered_cols += [f'{s}position', f'{s}dirtyprice', f'{s}PV', f'{s}P&L']

df = df[ordered_cols]
df

,stock1position,stock1dirtyprice,stock1PV,stock1P&L,stock2position,stock2dirtyprice,stock2PV,stock2P&L,stock3position,stock3dirtyprice,stock3PV,stock3P&L,stock4position,stock4dirtyprice,stock4PV,stock4P&L,stock5position,stock5dirtyprice,stock5PV,stock5P&L
EventID,,,,,,,,,,,,,,,,,,,,
1,-50,99.995,-4999.75,-4999.75,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.0000,0.0000,0.0000
2,-50,99.995,-4999.75,0.00,0,0.00,0.0,0.0,0,0.00,0.0,0.0,0,0.00,0.0,0.0,-60,106.8517,-6411.1000,-6411.1000
3,-50,99.995,-4999.75,0.00,0,0.00,0.0,0.0,-40,104.46,-4178.4,-4178.4,0,0.00,0.0,0.0,-60,106.8517,-6411.1000,0.0000
4,-50,99.995,-4999.75,0.00,0,0.00,0.0,0.0,-40,104.46,-4178.4,0.0,0,0.00,0.0,0.0,-80,106.0617,-8484.9333,-2073.8333
5,-30,100.265,-3007.95,1991.80,0,0.00,0.0,0.0,-40,104.46,-4178.4,0.0,0,0.00,0.0,0.0,-80,106.0617,-8484.9333,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,-120,100.315,-12037.80,0.00,240,100.75,24180.0,0.0,140,106.19,14866.6,0.0,720,92.97,66938.4,1635.4,-150,104.3617,-15654.2500,0.0000
147,-120,100.315,-12037.80,0.00,240,100.75,24180.0,0.0,140,106.19,14866.6,0.0,680,93.05,63274.0,-3664.4,-150,104.3617,-15654.2500,0.0000
148,-170,100.415,-17070.55,-5032.75,240,100.75,24180.0,0.0,140,106.19,14866.6,0.0,680,93.05,63274.0,0.0,-150,104.3617,-15654.2500,0.0000


In [31]:
# Build per-(desk, trader, bondID) present PV and total P&L
def build_combo_pv_pnl(events_df: pd.DataFrame, bonds_df: pd.DataFrame) -> pd.DataFrame:
    events_local = events_df.copy()
    bonds_local = bonds_df.copy()

    # Normalize bond column names to handle Frequency/frequency variations.
    bonds_local = bonds_local.rename(
        columns={
            "frequency": "Frequency",
            "coupon": "Coupon",
            "monthssincecoupon": "MonthsSinceCoupon",
        }
    )

    required_bond_cols = ["BondID", "Coupon", "Frequency", "MonthsSinceCoupon"]
    missing = [c for c in required_bond_cols if c not in bonds_local.columns]
    if missing:
        raise ValueError(f"Missing bond columns: {missing}")

    bonds_local = bonds_local[required_bond_cols].copy()
    df = events_local.merge(bonds_local, on="BondID", how="left")

    if df[["Coupon", "Frequency", "MonthsSinceCoupon"]].isna().any().any():
        raise ValueError("Missing bond metadata for one or more events")

    coupon_period_months = 12 / df["Frequency"]
    df["accrued_interest"] = (
        (df["Coupon"] / df["Frequency"])
        * (df["MonthsSinceCoupon"] / coupon_period_months)
        * 100
    )
    df["trade_dirty_price"] = df["CleanPrice"] + df["accrued_interest"]

    df["signed_qty"] = df["Quantity"].where(df["BuySell"].eq("BUY"), -df["Quantity"])

    latest_by_bond = (
        df.sort_values("EventID")
        .groupby("BondID", as_index=False)
        .tail(1)[["BondID", "CleanPrice", "accrued_interest"]]
        .rename(columns={"CleanPrice": "current_clean_price"})
    )
    latest_by_bond["current_dirty_price"] = (
        latest_by_bond["current_clean_price"] + latest_by_bond["accrued_interest"]
    )

    df = df.merge(latest_by_bond[["BondID", "current_dirty_price"]], on="BondID", how="left")

    df["present_pv"] = df["signed_qty"] * df["current_dirty_price"]
    df["event_trade_value"] = df["signed_qty"] * df["trade_dirty_price"]
    df["event_pnl"] = df["present_pv"] - df["event_trade_value"]

    combo_df = (
        df.groupby(["Desk", "Trader", "BondID"], as_index=False)
        .agg(
            present_pv=("present_pv", "sum"),
            total_pnl=("event_pnl", "sum"),
        )
        .rename(columns={"Desk": "desk", "Trader": "trader", "BondID": "bondID"})
        .sort_values(["desk", "trader", "bondID"])
        .reset_index(drop=True)
    )

    return combo_df

combo_df = build_combo_pv_pnl(events, bonds)
combo_df

,desk,trader,bondID,present_pv,total_pnl
0,HK,T_HK_1,BOND1,1010.400000,-21.5
1,HK,T_HK_1,BOND2,14994.000000,-11.4
2,HK,T_HK_1,BOND3,-1071.900000,-67.0
3,HK,T_HK_1,BOND4,2770.800000,-275.3
4,HK,T_HK_1,BOND5,-4186.133333,37.4
5,HK,T_HK_2,BOND1,-2020.800000,-1.9
6,HK,T_HK_2,BOND2,3998.400000,46.0
7,HK,T_HK_2,BOND3,-8575.200000,43.9
8,HK,T_HK_2,BOND4,23090.000000,-198.2
9,HK,T_HK_2,BOND5,-6279.200000,38.0


In [36]:
def query_bond_state(df, event_id, bond_id):
    # Extract bond number (e.g. BOND3 → 3)
    bond_idx = int(bond_id.replace("BOND", ""))
    s = f"stock{bond_idx}"
    
    # Safety check
    if event_id not in df.index:
        raise ValueError(f"Event {event_id} not found")
    
    row = df.loc[event_id]
    
    # Handle duplicate index edge case
    if isinstance(row, pd.DataFrame):
        row = row.iloc[-1]
    
    return {
        "position": row[f"{s}position"],
        "dirty_price": row[f"{s}dirtyprice"],
        "pv": row[f"{s}PV"]
    }


In [45]:
# Query functions with float conversion
def query_bond_state(df, event_id, bond_id):
    bond_idx = int(bond_id.replace("BOND", ""))
    s = f"stock{bond_idx}"
    if event_id not in df.index:
        raise ValueError(f"Event {event_id} not found")
    row = df.loc[event_id]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[-1]
    return {
        "position": float(row[f"{s}position"]),
        "dirty_price": float(row[f"{s}dirtyprice"]),
        "pv": float(row[f"{s}PV"])
    }

def query_bond_pnl(df, event_id, bond_id):
    bond_idx = int(bond_id.replace("BOND", ""))
    s = f"stock{bond_idx}"
    if event_id not in df.index:
        raise ValueError(f"Event {event_id} not found")
    return float(df.loc[event_id, f"{s}P&L"])

In [41]:
def get_trader_total_pnl(combo_df, trader):
    return combo_df[combo_df["trader"] == trader]["total_pnl"].sum()

In [49]:
event_id = 67
bond_state = query_bond_state(df, event_id, "BOND3")
bond3_pnl = query_bond_pnl(df, event_id, "BOND3")

print("Bond1 state at Event 67:", bond_state)
print("Bond3 PnL at Event 67:", bond3_pnl)

Bond1 state at Event 67: {'position': 60.0, 'dirty_price': 105.83, 'pv': 6349.8}
Bond3 PnL at Event 67: 0.0
